# Estate Monitor: one-time setup
Creates the storage and dashboard for the **Real-Time Estate Monitor** (see `docs/07-Real-Time-Estate-Monitor.md`).

What it does, in this workspace:
1. Creates the eventhouse **EstateMonitor** (and its KQL database) if it doesn't exist.
2. Runs `kql/01_Estate_Monitor_Tables.kql` and `kql/02_Estate_Monitor_Functions.kql`: the `RawEvents` table, its JSON mapping, retention and low-latency policies, and the typed views.
3. Creates or updates the Real-Time Dashboard **Fabric Estate Monitor**, already bound to the database.

Safe to rerun. Run it as a workspace Admin or Member. The Eventstream is created by hand afterwards (doc 07, step 3).


## Parameters

In [ ]:
EVENTHOUSE_NAME = "EstateMonitor"          # created in this workspace if it doesn't exist
DASHBOARD_NAME = "Fabric Estate Monitor"   # Real-Time Dashboard to create or update
DEPLOY_DASHBOARD = True

# Where to read the repo's kql/ and dashboards/ folders from:
SOURCE = "github"                          # "github" or "files"
REPO_RAW_ROOT = "https://raw.githubusercontent.com/<you>/<repo>/main"   # repo root, not /data
FILES_ROOT = "/lakehouse/default/Files/estate_monitor_repo"             # SOURCE = "files": upload kql/ and dashboards/ here

KQL_SCRIPTS = ["kql/01_Estate_Monitor_Tables.kql", "kql/02_Estate_Monitor_Functions.kql"]
DASHBOARD_FILE = "dashboards/Fabric_Estate_Monitor.RealTimeDashboard.json"


## Run

In [ ]:
import json, time, base64, re, requests
import sempy.fabric as fabric

try:
    nbu = notebookutils
except NameError:
    from notebookutils import mssparkutils as nbu

fab = fabric.FabricRestClient()
WS = fabric.get_workspace_id()

def read_source(rel):
    if SOURCE == "github":
        r = requests.get(f"{REPO_RAW_ROOT.rstrip('/')}/{rel}", timeout=60)
        r.raise_for_status()
        return r.text
    with open(f"{FILES_ROOT.rstrip('/')}/{rel}", encoding="utf-8") as f:
        return f.read()

def wait_lro(r):
    if r.status_code >= 400:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")
    if r.status_code != 202:
        return r
    loc = r.headers.get("Location")
    while True:
        time.sleep(int(r.headers.get("Retry-After", 3)))
        r = fab.get(loc)
        st = (r.json() or {}).get("status")
        if st in ("Succeeded", "Failed", "Cancelled"):
            if st != "Succeeded":
                raise RuntimeError(f"Operation {st}: {r.text[:500]}")
            return r

def list_items(kind):
    r = fab.get(f"v1/workspaces/{WS}/{kind}"); r.raise_for_status()
    return r.json().get("value", [])

# 1. Eventhouse (its default KQL database has the same name)
eh = next((e for e in list_items("eventhouses") if e["displayName"] == EVENTHOUSE_NAME), None)
if eh is None:
    print("Creating eventhouse", EVENTHOUSE_NAME)
    wait_lro(fab.post(f"v1/workspaces/{WS}/eventhouses", json={"displayName": EVENTHOUSE_NAME}))
    for _ in range(30):
        eh = next((e for e in list_items("eventhouses") if e["displayName"] == EVENTHOUSE_NAME), None)
        if eh: break
        time.sleep(5)
print("Eventhouse:", eh["id"])

db = None
for _ in range(30):
    dbs = [d for d in list_items("kqlDatabases") if (d.get("properties") or {}).get("parentEventhouseItemId") == eh["id"]]
    db = next((d for d in dbs if d["displayName"] == EVENTHOUSE_NAME), dbs[0] if dbs else None)
    if db and (db.get("properties") or {}).get("queryServiceUri"): break
    time.sleep(5)
QUERY_URI = db["properties"]["queryServiceUri"].rstrip("/")
DB_NAME, DB_ID = db["displayName"], db["id"]
print("KQL database:", DB_NAME, "| Query URI:", QUERY_URI)

# 2. Tables, mapping, policies and functions
def kusto_token():
    try:
        return nbu.credentials.getToken("kusto")
    except Exception:
        return nbu.credentials.getToken(QUERY_URI)

def split_commands(text):
    """Split a .kql script into control commands. Each command starts on a line beginning with '.';
    full-line comments at column 0 are dropped."""
    cmds, cur = [], []
    for line in text.splitlines():
        if line.startswith("//"):
            continue
        if line.startswith("."):
            if cur: cmds.append("\n".join(cur).strip())
            cur = [line]
        elif cur:
            cur.append(line)
    if cur: cmds.append("\n".join(cur).strip())
    return [c for c in cmds if c]

tok = kusto_token()
for script in KQL_SCRIPTS:
    for cmd in split_commands(read_source(script)):
        r = requests.post(f"{QUERY_URI}/v1/rest/mgmt", json={"db": DB_NAME, "csl": cmd}, timeout=120,
                          headers={"Authorization": f"Bearer {tok}", "Content-Type": "application/json; charset=utf-8"})
        first = cmd.splitlines()[0][:90]
        if r.ok:
            print("  ok   ", first)
        else:
            raise RuntimeError(f"Command failed: {first}\nHTTP {r.status_code}: {r.text[:800]}")

# 3. Real-Time Dashboard bound to this database
if DEPLOY_DASHBOARD:
    dash = json.loads(read_source(DASHBOARD_FILE))
    for ds in dash["dataSources"]:
        ds["clusterUri"], ds["database"], ds["name"] = QUERY_URI + "/", DB_ID, DB_NAME
    dash["title"] = DASHBOARD_NAME
    payload = base64.b64encode(json.dumps(dash).encode("utf-8")).decode()
    definition = {"parts": [{"path": "RealTimeDashboard.json", "payload": payload, "payloadType": "InlineBase64"}]}
    existing = next((d for d in list_items("kqlDashboards") if d["displayName"] == DASHBOARD_NAME), None)
    if existing:
        r = wait_lro(fab.post(f"v1/workspaces/{WS}/kqlDashboards/{existing['id']}/updateDefinition", json={"definition": definition}))
        print("Dashboard updated:", DASHBOARD_NAME)
    else:
        r = wait_lro(fab.post(f"v1/workspaces/{WS}/kqlDashboards", json={"displayName": DASHBOARD_NAME, "definition": definition}))
        print("Dashboard created:", DASHBOARD_NAME)

print("\nNext: create the Eventstream (doc 07, step 3) and point it at table RawEvents with mapping RawEvents_mapping.")
print("Collector settings -> EVENTHOUSE_QUERY_URI =", QUERY_URI, "| KQL_DATABASE =", DB_NAME)
